# AgeLens — 07 Governance Resolution

This notebook applies the evidence-backed governance resolutions after notebooks 01–06.

## Approved decisions applied

- **D-010:** Supplement pair `141.50225 / 0.090165` is canonical; Erratum pair remains sensitivity.
- **D-011:** Retain and flag `RIDAGEYR == 80`; never invent exact ages; require full/no-topcode reporting.
- **D-012:** Observed modern harmonized creatinine is canonical; `+0.11`, `+0.17`, `+0.23 mg/dL` are mandatory sensitivities.
- **D-013:** Approve Validation Check 1 tolerance and Check 2 first-run baseline.
- **D-014:** Normalize only the exact pandas/XPORT IBM-zero sentinel to `0.0`, with audit.

The notebook updates authoritative files **in place**, but creates timestamped backups first. Final scientific release remains disabled until notebook 08 regenerates canonical Supplement-primary outputs.

In [2]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import re
import shutil

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
RESOLUTION_DATE = "2026-07-22"
MARKER = "<!-- AGELENS GOVERNANCE RESOLUTION 2026-07-22 -->"

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)


def find_project_root(name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name.lower() == name.lower():
            return candidate
    raise FileNotFoundError(f"Could not find parent folder {name!r}.")


def find_unique(root: Path, filename: str) -> Path:
    matches = [
        p for p in root.rglob(filename)
        if "governance_backups" not in p.parts
    ]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {filename}; found {len(matches)}: {matches}"
        )
    return matches[0]


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


ROOT = find_project_root()
CONFIG_PATH = ROOT / "configs" / "agelens_config.json"
TABLES = ROOT / "results" / "tables"
LOGS = ROOT / "logs"
DOC_GOV = ROOT / "docs" / "governance"

DOCS = {
    "decision_log": find_unique(ROOT, "Decision_Log.md"),
    "gap_register": find_unique(ROOT, "Evidence_Gap_Register.md"),
    "replication_protocol": find_unique(ROOT, "Replication_Protocol.md"),
    "validation_protocol": find_unique(ROOT, "Validation_Protocol.md"),
}

for p in [LOGS, DOC_GOV]:
    p.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
for key, path in DOCS.items():
    print(f"  {key}: {path}")


Project root: <PROJECT_ROOT>
  decision_log: <PROJECT_ROOT>\docs\governance\Decision_Log.md
  gap_register: <PROJECT_ROOT>\docs\governance\Evidence_Gap_Register.md
  replication_protocol: <PROJECT_ROOT>\docs\methodology\Replication_Protocol.md
  validation_protocol: <PROJECT_ROOT>\docs\methodology\Validation_Protocol.md


## 1. Verify governance evidence

In [3]:
paths = {
    "status": TABLES / "05_validation_check_status.csv",
    "bioage": TABLES / "03_bioage_comparison.csv",
    "eg004": TABLES / "06_eg004_creatinine_sensitivity_summary.csv",
    "eg004_ref": TABLES / "06_eg004_reference_equivalence_check.csv",
    "sentinel": TABLES / "01_xpt_ibm_zero_sentinel_audit.csv",
}
missing = [p for p in paths.values() if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing governance evidence: {missing}")

status = pd.read_csv(paths["status"])
bioage = pd.read_csv(paths["bioage"])
eg004 = pd.read_csv(paths["eg004"])
eg004_ref = pd.read_csv(paths["eg004_ref"])
sentinel = pd.read_csv(paths["sentinel"])

token_files = sorted(TABLES.glob("04*source*token*audit*.csv"))
if len(token_files) != 1:
    raise RuntimeError(f"Expected one BioAge token audit: {token_files}")
tokens = pd.read_csv(token_files[0])

lookup = dict(zip(status["check"], status["status"]))
required = {
    "Check 1 — Age correlation vs BioAge": "PASS",
    "Check 2 — Cross-implementation agreement": "PASS_BASELINE_ESTABLISHED",
    "Check 3 — Bridging effectiveness": "PASS",
}
for name, expected in required.items():
    if lookup.get(name) != expected:
        raise RuntimeError(f"{name}: expected {expected}, found {lookup.get(name)!r}")

if not tokens["present"].astype(bool).all():
    raise RuntimeError("BioAge source-token audit failed.")
if not eg004_ref["pass"].astype(bool).all():
    raise RuntimeError("EG-004 zero-shift equivalence failed.")

wide = bioage.pivot(index="cycle", columns="agelens_variant", values="mae")
if not (wide["supplement"] < wide["erratum"]).all():
    raise RuntimeError("Supplement is not closer to BioAge in every cycle.")

supp = bioage.loc[bioage["agelens_variant"].eq("supplement")].copy()
if not supp["mae"].lt(0.10).all():
    raise RuntimeError("Supplement MAE exceeds D-013 bound.")
if not (supp["pearson"].ge(0.999999).all() and supp["spearman"].ge(0.999999).all()):
    raise RuntimeError("Supplement correlation baseline failed.")

eg = eg004.loc[
    eg004["sample"].eq("all_harmonized_complete_case")
    & eg004["domain"].eq("pooled")
    & eg004["formula_variant"].eq("supplement")
    & eg004["scenario"].isin(["nhanes3_bias_low","nhanes3_bias_mid","nhanes3_bias_high"])
].sort_values("creatinine_shift_mg_dL")
if len(eg) != 3:
    raise RuntimeError("Expected three EG-004 Supplement scenarios.")
effects = eg["weighted_mean_delta_years"].to_numpy(float)
if not (1.0 <= effects[0] <= 1.1 and 1.5 <= effects[1] <= 1.7 and 2.0 <= effects[2] <= 2.2):
    raise RuntimeError(f"Unexpected EG-004 effects: {effects}")

sentinel_total = int(sentinel["replacement_count"].sum())
if sentinel_total <= 0 or len(sentinel.loc[sentinel["column"].eq("WTSAF2YR")]) != 2:
    raise RuntimeError("XPT sentinel audit is incomplete.")

summary = pd.DataFrame([
    {"evidence":"Check 1","result":lookup["Check 1 — Age correlation vs BioAge"]},
    {"evidence":"Check 2","result":lookup["Check 2 — Cross-implementation agreement"]},
    {"evidence":"Check 3","result":lookup["Check 3 — Bridging effectiveness"]},
    {"evidence":"BioAge token audit","result":f"{int(tokens['present'].sum())}/{len(tokens)}"},
    {"evidence":"Supplement MAE","result":f"{supp['mae'].min():.6f}–{supp['mae'].max():.6f}"},
    {"evidence":"EG-004 effects","result":", ".join(f"{x:.6f}" for x in effects)},
    {"evidence":"XPT sentinel replacements","result":f"{sentinel_total:,}"},
])
display(summary)
print("✅ Governance evidence verified.")


,evidence,result
0,Check 1,PASS
1,Check 2,PASS_BASELINE_ESTABLISHED
2,Check 3,PASS
3,BioAge token audit,16/16
4,Supplement MAE,0.049726–0.050348
5,EG-004 effects,"1.024544, 1.583386, 2.142228"
6,XPT sentinel replacements,"80,902"


✅ Governance evidence verified.


## 2. Build authoritative document updates

In [4]:
EXPECTED = {
    "decision_log": "1.6",
    "gap_register": "0.18",
    "replication_protocol": "1.2",
    "validation_protocol": "1.1",
}
NEW = {
    "decision_log": "1.7",
    "gap_register": "0.19",
    "replication_protocol": "1.3",
    "validation_protocol": "1.2",
}


def doc_version(text: str) -> str:
    m = re.search(r"(?m)^\|\s*Version\s*\|\s*([^|]+?)\s*\|$", text)
    if not m:
        raise RuntimeError("Version row not found.")
    return m.group(1).strip()


def set_field(text: str, field: str, value: str) -> str:
    pattern = re.compile(rf"(?m)^(\|\s*{re.escape(field)}\s*\|\s*)([^|]*?)(\s*\|)$")
    out, n = pattern.subn(lambda m: f"{m.group(1)}{value}{m.group(3)}", text, count=1)
    if n != 1:
        raise RuntimeError(f"Could not set {field}.")
    return out


def add_revision(text: str, row: str) -> str:
    anchor = "*This file is a single authoritative document"
    i = text.find(anchor)
    if i < 0:
        raise RuntimeError("Authoritative-document marker missing.")
    if row in text:
        return text
    return text[:i].rstrip() + "\n" + row + "\n\n" + text[i:]


def section_bounds(text: str, heading: str):
    start = text.find(heading)
    if start < 0:
        raise RuntimeError(f"Heading not found: {heading}")
    m = re.search(r"(?m)^###\s+", text[start + len(heading):])
    end = len(text) if not m else start + len(heading) + m.start()
    return start, end


def set_gap_status(text: str, gap: str, status_text: str, evidence_text: str) -> str:
    start, end = section_bounds(text, f"### {gap}")
    section = text[start:end]
    pattern = re.compile(r"(?m)^(\|\s*Review Status\s*\|\s*)([^|]*?)(\s*\|)$")
    section, n = pattern.subn(lambda m: f"{m.group(1)}{status_text}{m.group(3)}", section, count=1)
    if n != 1:
        raise RuntimeError(f"Review Status missing in {gap}.")
    evidence_row = f"| Resolution Evidence | {evidence_text} |\n"
    if "| Resolution Evidence |" not in section:
        pos = re.search(r"(?m)^\|\s*Review Status\s*\|", section).start()
        section = section[:pos] + evidence_row + section[pos:]
    return text[:start] + section + text[end:]


def replace_from_heading(text: str, heading: str, replacement: str) -> str:
    i = text.find(heading)
    if i < 0:
        raise RuntimeError(f"Heading not found: {heading}")
    return text[:i] + replacement.rstrip() + "\n"

original = {k:p.read_text(encoding="utf-8") for k,p in DOCS.items()}
for key, text in original.items():
    if MARKER in text:
        raise RuntimeError(f"{key} already resolved; do not rerun.")
    if doc_version(text) != EXPECTED[key]:
        raise RuntimeError(f"{key}: expected v{EXPECTED[key]}, found v{doc_version(text)}")

# Decision Log
D = set_field(original["decision_log"], "Version", NEW["decision_log"])
D = set_field(D, "Last Updated", RESOLUTION_DATE)
D = add_revision(D, "| 1.7 | 2026-07-22 | Approved D-010 through D-014; dispositioned EG-002, EG-004, EG-010, and EG-014 after completed validation, source inspection, EG-004 sensitivity, and XPT ingestion audit. |")
entries = f"""\n\n{MARKER}\n\n---\n\n### D-010\n\n| Field | Value |\n| --- | --- |\n| Title | Canonical final conversion pair |\n| Description | Adopt Supplement 1's `141.50225 / 0.090165` pair as canonical. Retain `141.50 / 0.09165` as a named sensitivity. Hybrid pairs are prohibited. |\n| Supporting Evidence | Direct BioAge source inspection; Supplement MAE {supp['mae'].min():.6f}–{supp['mae'].max():.6f} years versus approximately 1.6 years for Erratum; Pearson/Spearman = 1.0. |\n| Evidence Level | E4 plus E1/E3 corroboration |\n| Confidence Rating | High |\n| Reviewer | Project owner |\n| Status | Approved |\n| Date | 2026-07-22 |\n| Related Evidence Gaps | EG-010 — Closed |\n| Notes | The remaining approximately 0.05-year difference reflects rounded published coefficients versus BioAge's higher-precision coefficients. |\n\n---\n\n### D-011\n\n| Field | Value |\n| --- | --- |\n| Title | Handling of age top-coding at 80 |\n| Description | Retain `RIDAGEYR == 80`, set `age_topcoded = TRUE`, never invent exact ages, and report full-sample plus no-topcode sensitivities. |\n| Supporting Evidence | Official NHANES top-coding documentation and completed full/no-topcode validation. |\n| Evidence Level | E2 plus direct V1 validation |\n| Confidence Rating | High for handling policy |\n| Reviewer | Project owner |\n| Status | Approved |\n| Date | 2026-07-22 |\n| Related Evidence Gaps | EG-014 — Closed as documented limitation |\n| Notes | Top-coded records are excluded from the no-topcode face-validity correlation but retained in BioAge agreement checks. |\n\n---\n\n### D-012\n\n| Field | Value |\n| --- | --- |\n| Title | Creatinine training-scale policy |\n| Description | Use observed modern harmonized creatinine in the canonical replication. Report `+0.11`, `+0.17`, and `+0.23 mg/dL` shifts as mandatory sensitivities. |\n| Supporting Evidence | Notebook 06 produced Supplement shifts of {effects[0]:.6f}, {effects[1]:.6f}, and {effects[2]:.6f} years. |\n| Evidence Level | E1 plus governed sensitivity analysis |\n| Confidence Rating | Moderate |\n| Reviewer | Project owner |\n| Status | Approved |\n| Date | 2026-07-22 |\n| Related Evidence Gaps | EG-004 — Closed as accepted and quantified limitation |\n| Notes | No compensating shift becomes canonical without a future superseding Decision. |\n\n---\n\n### D-013\n\n| Field | Value |\n| --- | --- |\n| Title | V1 validation acceptance criteria |\n| Description | Approve Check 1 `|Δr| < 0.02`. Check 2 passes when Supplement MAE is below 0.10 years and Pearson/Spearman are at least 0.999999 in each cycle. |\n| Supporting Evidence | Completed Checks 1–4 and independent R `survey` verification. |\n| Evidence Level | Direct V1 validation evidence |\n| Confidence Rating | High for deterministic regression testing |\n| Reviewer | Project owner |\n| Status | Approved |\n| Date | 2026-07-22 |\n| Related Evidence Gaps | Supports closure of EG-002 and EG-010 |\n| Notes | Check 4 remains a documented limitation; Little's MCAR was not run. |\n\n---\n\n### D-014\n\n| Field | Value |\n| --- | --- |\n| Title | Exact pandas/XPORT IBM-zero sentinel normalization |\n| Description | Convert only exact `5.397605346934028e-79` values to `0.0` immediately after XPT read and audit every replacement. General near-zero thresholding is prohibited. |\n| Supporting Evidence | Ingestion audit documented {sentinel_total:,} replacements, including both fasting-weight files. |\n| Evidence Level | Direct implementation audit |\n| Confidence Rating | High |\n| Reviewer | Project owner |\n| Status | Approved |\n| Date | 2026-07-22 |\n| Related Evidence Gaps | — |\n| Notes | Parser correction, not biological imputation. |\n"""
D = D.rstrip() + entries

# Evidence Gap Register
G = set_field(original["gap_register"], "Version", NEW["gap_register"])
G = set_field(G, "Last Updated", RESOLUTION_DATE)
G = add_revision(G, "| 0.19 | 2026-07-22 | Closed EG-002 after direct BioAge source inspection; closed EG-004, EG-010, and EG-014 through D-012, D-010, and D-011. |")
G = set_gap_status(G, "EG-002", "**Closed — direct source inspection completed**, 2026-07-22.", "Notebook 04 source-token audit: all required coefficients and constants present.")
G = set_gap_status(G, "EG-004", "**Closed — accepted and quantified limitation; converted to D-012**, 2026-07-22.", f"Notebook 06 Supplement shifts: {effects[0]:.6f}, {effects[1]:.6f}, {effects[2]:.6f} years.")
G = set_gap_status(G, "EG-010", "**Closed — converted to D-010**, 2026-07-22.", f"Supplement MAE {supp['mae'].min():.6f}–{supp['mae'].max():.6f}; Pearson/Spearman = 1.0.")
G = set_gap_status(G, "EG-014", "**Closed — accepted limitation; converted to D-011**, 2026-07-22.", "Full and no-topcode validation completed; retain/flag/report policy approved.")
G = G.rstrip() + "\n\n" + MARKER + "\n"

# Replication Protocol
R = set_field(original["replication_protocol"], "Version", NEW["replication_protocol"])
R = set_field(R, "Last Updated", RESOLUTION_DATE)
R = set_field(R, "Status", "Approved for canonical V1 rebuild")
R = add_revision(R, "| 1.3 | 2026-07-22 | Incorporated D-010 through D-014 and removed the former Core-gap blockers after completed governance resolution. |")
old_formula = "PhenotypicAge = 141.50 + ln( -0.00553 · ln(1 - M) ) / 0.09165"
new_formula = "PhenotypicAge = 141.50225 + ln( -0.00553 · ln(1 - M) ) / 0.090165"
if old_formula not in R:
    raise RuntimeError("Old formula line not found in Replication Protocol.")
R = R.replace(old_formula, new_formula, 1)
start_note = R.find("**Note on constant pairing (added 2026-07-19):**")
end_note = R.find("Apply the exact formula specified", start_note)
if start_note < 0 or end_note < 0:
    raise RuntimeError("Constant-pair note not found.")
R = R[:start_note] + "**Canonical constant pair (D-010):** Use `141.50225 / 0.090165`. Retain the Erratum pair only as sensitivity; hybrid pairs are prohibited.\n\n" + R[end_note:]
new_tail = f"""## 12. Governance Status\n\nValidation Checks 1–4 are complete. D-010 through D-014 were approved on 2026-07-22. EG-002, EG-004, EG-010, and EG-014 are closed; no Core Evidence Gap remains open for the governed V1 replication path.\n\nMandatory rules:\n\n- D-010: Supplement pair is canonical; Erratum pair is sensitivity.\n- D-011: retain and flag age-topcoded records; report full/no-topcode results.\n- D-012: observed modern creatinine is canonical; +0.11/+0.17/+0.23 mg/dL shifts are mandatory sensitivities.\n- D-014: normalize only the exact XPT IBM-zero sentinel and audit replacements.\n\nFinal release is still blocked until canonical Supplement-primary outputs are regenerated and regression-tested.\n\n## 13. Next Steps\n\n1. Run `08_canonical_output_rebuild.ipynb`.\n2. Preserve all named sensitivities.\n3. Verify D-013 regression checks.\n4. Enable final V1 outputs only after the canonical rebuild gate passes.\n\n{MARKER}\n"""
R = replace_from_heading(R, "## 12. Governance Status", new_tail)

# Validation Protocol
V = set_field(original["validation_protocol"], "Version", NEW["validation_protocol"])
V = set_field(V, "Last Updated", RESOLUTION_DATE)
V = set_field(V, "Status", "Approved — V1 checks completed")
V = add_revision(V, "| 1.2 | 2026-07-22 | Checks 1–4 completed; D-013 approved Check 1 tolerance and Check 2 baseline; Core gaps dispositioned. |")
old_phrase = "(proposed: |Δr| < 0.02) — exact tolerance to be confirmed and logged as a Decision before first use"
if old_phrase not in V:
    raise RuntimeError("Validation Check 1 provisional phrase not found.")
V = V.replace(old_phrase, "(D-013 approved: |Δr| < 0.02)", 1)
new_v_tail = f"""## 7. Reporting\n\nAll four checks were executed and documented in `docs/methodology/Validation_Report_Draft.md`.\n\n- Check 1: PASS.\n- Check 2: PASS — baseline established.\n- Check 3: PASS.\n- Check 4: PASS WITH DOCUMENTED LIMITATION.\n\n## 8. Resolved Open Items\n\n1. D-013 approved Check 1 `|Δr| < 0.02`.\n2. D-013 established Check 2: Supplement MAE < 0.10 years and Pearson/Spearman >= 0.999999.\n3. BioAge direct source inspection completed; EG-002 closed.\n4. EG-004, EG-010, and EG-014 were dispositioned through D-012, D-010, and D-011.\n\n## 9. Next Steps\n\nRegenerate canonical Supplement-primary outputs. Final release remains disabled until that rebuild and its D-013 regression checks pass.\n\n{MARKER}\n"""
V = replace_from_heading(V, "## 7. Reporting", new_v_tail)

UPDATED = {"decision_log":D, "gap_register":G, "replication_protocol":R, "validation_protocol":V}
for key, text in UPDATED.items():
    if doc_version(text) != NEW[key] or MARKER not in text:
        raise RuntimeError(f"Constructed {key} failed version/marker checks.")
print("✅ Authoritative updates constructed.")


✅ Authoritative updates constructed.


## 3. Update configuration, back up, and write

In [5]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
expected_open = {"EG-004","EG-010","EG-014"}
if set(config["governance"].get("open_core_evidence_gaps", [])) != expected_open:
    raise RuntimeError("Config open Core gaps do not match expected pre-resolution state.")

config["project"]["software_version"] = "0.2.0"
config["project"]["status"] = "governance_resolved_pending_canonical_rebuild"
config["project"]["final_scientific_results_allowed"] = False
config["governance"]["open_core_evidence_gaps"] = []
config["governance"]["closed_2026_07_22"] = ["EG-002","EG-004","EG-010","EG-014"]
config["governance"].setdefault("decisions", {}).update({
    "formula_constants":"D-010",
    "age_topcoding":"D-011",
    "creatinine_training_scale":"D-012",
    "validation_acceptance":"D-013",
    "xpt_zero_normalization":"D-014",
})
config["formula"]["primary_variant"] = "supplement"
config["formula"]["canonical_variant"] = "supplement"
config["formula"]["sensitivity_variants"] = ["erratum"]
config["age_topcoding_policy"] = {
    "decision":"D-011","retain":True,"flag":"age_topcoded",
    "impute_exact_age":False,"mandatory_no_topcode_sensitivity":True,
}
config["creatinine_training_scale_policy"] = {
    "decision":"D-012","canonical_shift_mg_dL":0.0,
    "mandatory_sensitivity_shifts_mg_dL":[0.11,0.17,0.23],
}
config["validation"] = {
    "decision":"D-013","check_1_max_absolute_delta_r":0.02,
    "check_2_max_mae_years":0.10,"check_2_min_pearson":0.999999,
    "check_2_min_spearman":0.999999,
    "supplement_baseline_mae_by_cycle":{row["cycle"]:float(row["mae"]) for _,row in supp.iterrows()},
}
config.setdefault("ingestion", {})["xpt_ibm_zero_normalization"] = {
    "decision":"D-014","sentinel_value":5.397605346934028e-79,
    "replacement_value":0.0,"exact_match_only":True,"audit_required":True,
}
config["release_gates"] = {
    "governance_core_gaps_resolved":True,
    "validation_completed":True,
    "canonical_outputs_regenerated":False,
    "canonical_regression_checks_passed":False,
    "mortality_analysis_authorized":False,
    "final_scientific_results_allowed":False,
}
config_text = json.dumps(config, indent=2, ensure_ascii=False) + "\n"

timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
backup_root = LOGS / "governance_backups" / timestamp
backup_root.mkdir(parents=True, exist_ok=False)

audit=[]
all_targets = {**DOCS, "config":CONFIG_PATH}
for key,path in all_targets.items():
    backup = backup_root / f"{key}__{path.name}"
    shutil.copy2(path, backup)
    audit.append({
        "artifact":key,"path":str(path.relative_to(ROOT)),
        "backup":str(backup.relative_to(ROOT)),
        "sha256_before":sha256(path),"sha256_backup":sha256(backup),
    })

for key,path in DOCS.items():
    path.write_text(UPDATED[key], encoding="utf-8")
CONFIG_PATH.write_text(config_text, encoding="utf-8")

for row in audit:
    path = ROOT / row["path"]
    row["sha256_after"] = sha256(path)
    if row["sha256_before"] != row["sha256_backup"]:
        raise RuntimeError(f"Backup hash mismatch: {row['artifact']}")
    if row["sha256_before"] == row["sha256_after"]:
        raise RuntimeError(f"Artifact did not change: {row['artifact']}")

audit_df = pd.DataFrame(audit)
audit_path = TABLES / "07_governance_resolution_audit.csv"
audit_df.to_csv(audit_path, index=False)
display(audit_df)
print(f"Backup: {backup_root}")


,artifact,path,backup,sha256_before,sha256_backup,sha256_after
0,decision_log,docs\governance\Decision_Log.md,logs\governance_backups\20260722T155150Z\decis...,27b564cd5cf8784f00f07dc085b97b143bf58151724c52...,27b564cd5cf8784f00f07dc085b97b143bf58151724c52...,f93c540bc24ee3e5ed8d044f9f9df2a3db2b16c1e1c245...
1,gap_register,docs\governance\Evidence_Gap_Register.md,logs\governance_backups\20260722T155150Z\gap_r...,eee7023dee1ee1c954be0d1b5987682c77fdb2600d7b5e...,eee7023dee1ee1c954be0d1b5987682c77fdb2600d7b5e...,9e49bdd6a69424b50b7787bde48e93d298574892193693...
2,replication_protocol,docs\methodology\Replication_Protocol.md,logs\governance_backups\20260722T155150Z\repli...,f0519342372249356b5f0238ee071dbb82c525399caf8d...,f0519342372249356b5f0238ee071dbb82c525399caf8d...,c1d6adcfb11bf4ec122005eb55bd1213c3715a8fa0a183...
3,validation_protocol,docs\methodology\Validation_Protocol.md,logs\governance_backups\20260722T155150Z\valid...,b51b823cc6c3ab3ae08f1dcb64feb6a2d6188245572b19...,b51b823cc6c3ab3ae08f1dcb64feb6a2d6188245572b19...,b1c94663093c12decd3eeefdf618dc0e8954b2325f77f4...
4,config,configs\agelens_config.json,logs\governance_backups\20260722T155150Z\confi...,9469168edca272e7bc41f4e6dff1948ce071158f3084fc...,9469168edca272e7bc41f4e6dff1948ce071158f3084fc...,15f5ffca2aa6cdf81c23acccffa917597594efe95b41ed...


Backup: <PROJECT_ROOT>\logs\governance_backups\20260722T155150Z


## 4. Final cross-document checks

In [6]:
reloaded = {k:p.read_text(encoding="utf-8") for k,p in DOCS.items()}
cfg = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
checks=[]
for decision in ["D-010","D-011","D-012","D-013","D-014"]:
    checks.append({"check":f"{decision} in Decision Log","pass":f"### {decision}" in reloaded["decision_log"]})
for gap in ["EG-002","EG-004","EG-010","EG-014"]:
    a,b = section_bounds(reloaded["gap_register"], f"### {gap}")
    checks.append({"check":f"{gap} closed","pass":"Closed" in reloaded["gap_register"][a:b]})
checks += [
    {"check":"Supplement formula canonical","pass":"PhenotypicAge = 141.50225" in reloaded["replication_protocol"] and "/ 0.090165" in reloaded["replication_protocol"]},
    {"check":"No Core gaps in config","pass":cfg["governance"]["open_core_evidence_gaps"] == []},
    {"check":"Supplement primary in config","pass":cfg["formula"]["primary_variant"] == "supplement"},
    {"check":"Final release still blocked","pass":not cfg["project"]["final_scientific_results_allowed"] and not cfg["release_gates"]["canonical_outputs_regenerated"]},
]
checks_df = pd.DataFrame(checks)
if not checks_df["pass"].all():
    display(checks_df.loc[~checks_df["pass"]])
    raise RuntimeError("Governance consistency check failed.")
checks_path = TABLES / "07_governance_consistency_checks.csv"
checks_df.to_csv(checks_path, index=False)
display(checks_df)

report_path = DOC_GOV / "Governance_Resolution_Report.md"
report_path.write_text(f"""# AgeLens Governance Resolution Report\n\nApplied: {RESOLUTION_DATE}\n\nApproved: D-010 through D-014.\n\nClosed: EG-002, EG-004, EG-010, EG-014.\n\nNo Core Evidence Gap remains open. Validation is complete. Final scientific release remains disabled until canonical Supplement-primary outputs are regenerated and regression-tested.\n\nBackup: `{backup_root.relative_to(ROOT)}`\n""", encoding="utf-8")
metadata_path = LOGS / "07_governance_resolution_metadata.json"
metadata_path.write_text(json.dumps({
    "created_at_utc":datetime.now(timezone.utc).isoformat(),
    "decisions":["D-010","D-011","D-012","D-013","D-014"],
    "closed_gaps":["EG-002","EG-004","EG-010","EG-014"],
    "backup_root":str(backup_root.relative_to(ROOT)),
    "mortality_data_used":False,
    "canonical_outputs_regenerated":False,
    "final_scientific_results_allowed":False,
    "next_notebook":"08_canonical_output_rebuild.ipynb",
}, indent=2), encoding="utf-8")

print("✅ Governance resolution completed.")
print("No Core Evidence Gap remains open.")
print("Canonical output rebuild is still required.")
print("Mortality data were not used.")
print("Final scientific results remain disabled.")


,check,pass
0,D-010 in Decision Log,True
1,D-011 in Decision Log,True
2,D-012 in Decision Log,True
3,D-013 in Decision Log,True
4,D-014 in Decision Log,True
5,EG-002 closed,True
6,EG-004 closed,True
7,EG-010 closed,True
8,EG-014 closed,True
9,Supplement formula canonical,True


✅ Governance resolution completed.
No Core Evidence Gap remains open.
Canonical output rebuild is still required.
Mortality data were not used.
Final scientific results remain disabled.
